# Continuous Probability Distributions

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why continuous RVs use **PDFs** (not PMFs) and why $P(X = x) = 0$.
2. Compute probabilities as areas under the PDF: $P(a \leq X \leq b) = \int_a^b f(x)\,dx$.
3. Derive properties of the **Uniform**, **Normal**, **Exponential**, **Gamma**, and **Beta** distributions.
4. Apply the **68-95-99.7 rule** and **z-scores** for the Normal distribution.
5. Recognise relationships between distributions (Exponential ↔ Poisson, Gamma generalises Exponential, Beta as prior for Binomial).

## Prerequisites

- [01_discrete_distributions.ipynb](01_discrete_distributions.ipynb) — discrete distributions, PMF, `scipy.stats` pattern
- [Math foundations](../00_prerequisites/02_math_foundations.ipynb) — integrals, exponentials, logarithms

In [ ]:
import sys, os, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "sudo apt-get update -qq && sudo apt-get install -y -qq "
        "libcairo2-dev libpango1.0-dev && pip install -q manim ipython==8.21.0"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)
        if not self.has_latex:
            print("⚠ LaTeX not found — MathTex will fall back to Text().")

    def apply_manim_config(self):
        from manim import config as mcfg

        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text

        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  ✓ media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)

---

## 1. From PMF to PDF — The Continuous Setting

For a discrete RV, the PMF gives $P(X = x)$ for each value. But for a continuous RV (like a measurement of height, time, or temperature), there are **uncountably many** possible values. If we assigned positive probability to each one, the total would be infinite — violating the axioms.

The resolution: for continuous RVs, $P(X = x) = 0$ for every specific $x$. Instead, probability is described by a **density**.

> **Definition (PDF).** A continuous random variable $X$ has *probability density function* $f(x)$ if:
>
> $$P(a \leq X \leq b) = \int_a^b f(x)\,dx$$
>
> with $f(x) \geq 0$ everywhere and $\int_{-\infty}^{\infty} f(x)\,dx = 1$.

**Key difference from PMF:** The PDF value $f(x)$ is **not** a probability — it's a *density*. It can be greater than 1! The probability lives in the **area** under the curve.

The CDF is the same concept as in the discrete case, now defined by integration:

$$F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t)\,dt \qquad \text{and} \qquad f(x) = F'(x)$$

Expectation and variance also generalise:

$$E[X] = \int_{-\infty}^{\infty} x\, f(x)\,dx \qquad \text{Var}(X) = \int_{-\infty}^{\infty} (x - \mu)^2 f(x)\,dx$$

---

## 2. The Uniform Distribution

The simplest continuous distribution: every value in an interval $[a, b]$ is equally likely.

> **Definition.** $X \sim \text{Uniform}(a, b)$ has PDF:
>
> $$f(x) = \frac{1}{b-a} \quad \text{for } a \leq x \leq b, \qquad 0 \text{ otherwise}$$

$$E[X] = \frac{a+b}{2}, \qquad \text{Var}(X) = \frac{(b-a)^2}{12}$$

The CDF is a straight line from 0 to 1 across $[a, b]$:

$$F(x) = \frac{x - a}{b - a} \quad \text{for } a \leq x \leq b$$

In [ ]:
# Uniform(0, 1): PDF, CDF, and histogram of samples
a, b = 0, 1
x = np.linspace(-0.5, 1.5, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# PDF
axes[0].plot(x, stats.uniform(a, b - a).pdf(x), linewidth=2.5)
axes[0].set_title("PDF")
axes[0].set_ylabel("f(x)")
axes[0].set_ylim(0, 1.5)

# CDF
axes[1].plot(x, stats.uniform(a, b - a).cdf(x), linewidth=2.5)
axes[1].set_title("CDF")
axes[1].set_ylabel("F(x)")

# Samples histogram
samples = rng.uniform(a, b, size=10_000)
axes[2].hist(samples, bins=40, density=True, alpha=0.7, edgecolor="white")
axes[2].axhline(
    1 / (b - a), color="black", linestyle="--", linewidth=1.5, label="f(x) = 1"
)
axes[2].set_title("10,000 Samples")
axes[2].legend(fontsize=9)

for ax in axes:
    ax.set_xlabel("x")

plt.suptitle("Uniform(0, 1)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. The Normal (Gaussian) Distribution

The most important distribution in statistics. It appears everywhere — partly because of the Central Limit Theorem (next notebook), and partly because many natural measurements are approximately Normal.

> **Definition.** $X \sim \text{Normal}(\mu, \sigma^2)$ (or $\mathcal{N}(\mu, \sigma^2)$) has PDF:
>
> $$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right), \quad x \in \mathbb{R}$$

$$E[X] = \mu, \qquad \text{Var}(X) = \sigma^2$$

The **standard Normal** is $Z \sim \mathcal{N}(0, 1)$. Any Normal can be standardised: $Z = (X - \mu) / \sigma$.

### The 68-95-99.7 rule

For any Normal distribution:
- $P(|X - \mu| \leq \sigma) \approx 0.683$ — about 68% within 1 SD
- $P(|X - \mu| \leq 2\sigma) \approx 0.954$ — about 95% within 2 SDs
- $P(|X - \mu| \leq 3\sigma) \approx 0.997$ — about 99.7% within 3 SDs

Compare with Chebyshev's inequality (Module 01, notebook 06): Chebyshev gives $\leq 1/k^2$ for any distribution. For the Normal, the actual probabilities are much tighter.

In [ ]:
# The Normal PDF for different parameters
fig, ax = plt.subplots(figsize=(9, 5))
x = np.linspace(-6, 10, 400)

params = [
    (0, 1, "N(0, 1)"),
    (0, 2, "N(0, 4)"),
    (2, 1, "N(2, 1)"),
    (4, 0.5, "N(4, 0.25)"),
]
for mu, sigma, label in params:
    ax.plot(x, stats.norm(mu, sigma).pdf(x), linewidth=2, label=label)

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Normal Distribution: Different μ and σ")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 68-95-99.7 rule — visualise with shaded regions
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(-4, 4, 500)
y = stats.norm.pdf(x)

ax.plot(x, y, "k-", linewidth=2)

# Shade regions
fills = [
    (-3, 3, "#619CFF", 0.15, "99.7% (3σ)"),
    (-2, 2, "#00BA38", 0.20, "95.4% (2σ)"),
    (-1, 1, "#F8766D", 0.30, "68.3% (1σ)"),
]
for lo, hi, col, alpha, label in fills:
    mask = (x >= lo) & (x <= hi)
    ax.fill_between(x[mask], y[mask], color=col, alpha=alpha, label=label)

ax.set_xlabel("z (standard deviations from mean)")
ax.set_ylabel("f(z)")
ax.set_title("The 68-95-99.7 Rule for the Standard Normal")
ax.legend(loc="upper right", fontsize=10)
ax.set_xticks(range(-4, 5))
plt.tight_layout()
plt.show()

# Exact values
for k in [1, 2, 3]:
    prob = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"P(|Z| ≤ {k}) = {prob:.6f}  ({prob:.1%})")

---

## 4. The Exponential Distribution — Waiting Times

If events occur according to a Poisson process at rate $\lambda$ per unit time, the **time between events** follows an Exponential distribution. It is the continuous analogue of the Geometric distribution.

> **Definition.** $X \sim \text{Exponential}(\lambda)$ has PDF:
>
> $$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

$$E[X] = \frac{1}{\lambda}, \qquad \text{Var}(X) = \frac{1}{\lambda^2}$$

Like the Geometric, the Exponential is **memoryless**: $P(X > s + t \mid X > s) = P(X > t)$. It is the *only* continuous distribution with this property.

### Connection to Poisson

If the inter-arrival times are $\text{Exponential}(\lambda)$, then the number of arrivals in time $t$ is $\text{Poisson}(\lambda t)$. The two distributions are "duals".

In [ ]:
# Exponential PDF for different rates
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(0, 5, 300)

for lam, color in [
    (0.5, "#F8766D"),
    (1.0, "#00BA38"),
    (2.0, "#619CFF"),
    (4.0, "#F564E3"),
]:
    ax.plot(
        x,
        stats.expon(scale=1 / lam).pdf(x),
        linewidth=2,
        color=color,
        label=f"Exp(λ={lam}), E[X]={1 / lam}",
    )

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Exponential Distribution: Different Rates")
ax.legend(fontsize=9)
ax.set_ylim(0, 4.5)
plt.tight_layout()
plt.show()

---

## 5. The Gamma Distribution — Generalising the Exponential

The Exponential is the time until the **first** Poisson event. What about the time until the $\alpha$-th event? That's the Gamma distribution.

> **Definition.** $X \sim \text{Gamma}(\alpha, \beta)$ has PDF:
>
> $$f(x) = \frac{\beta^\alpha}{\Gamma(\alpha)} x^{\alpha-1} e^{-\beta x}, \quad x > 0$$
>
> where $\Gamma(\alpha) = \int_0^\infty t^{\alpha-1} e^{-t}\,dt$ is the Gamma function ($\Gamma(n) = (n-1)!$ for integers).

$$E[X] = \frac{\alpha}{\beta}, \qquad \text{Var}(X) = \frac{\alpha}{\beta^2}$$

**Special cases:**
- $\text{Gamma}(1, \lambda) = \text{Exponential}(\lambda)$
- $\text{Gamma}(n/2, 1/2) = \chi^2(n)$ (chi-squared, used in hypothesis testing — Module 05)

In [ ]:
# Gamma PDFs for different shape parameters
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(0.01, 15, 300)

params = [(1, 1, "α=1 (= Exp)"), (2, 1, "α=2"), (5, 1, "α=5"), (10, 1, "α=10")]
for alpha, beta, label in params:
    ax.plot(x, stats.gamma(a=alpha, scale=1 / beta).pdf(x), linewidth=2, label=label)

ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("Gamma(α, β=1): Shape Changes with α")
ax.legend()
plt.tight_layout()
plt.show()

As $\alpha$ grows, the Gamma becomes more symmetric and more bell-shaped — it starts to look Normal. This is another instance of the Central Limit Theorem at work.

---

## 6. The Beta Distribution — Probabilities of Probabilities

The Beta distribution is defined on $[0, 1]$, making it perfect for modelling **probabilities** or **proportions**. It will play a central role in Bayesian statistics (Module 07) as the **conjugate prior** for the Binomial.

> **Definition.** $X \sim \text{Beta}(\alpha, \beta)$ has PDF:
>
> $$f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}, \quad 0 < x < 1$$
>
> where $B(\alpha, \beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}$ is the Beta function.

$$E[X] = \frac{\alpha}{\alpha + \beta}, \qquad \text{Var}(X) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$$

**Special cases:**
- $\text{Beta}(1, 1) = \text{Uniform}(0, 1)$ — no prior information
- $\alpha > \beta$: distribution shifted toward 1
- $\alpha < \beta$: shifted toward 0
- $\alpha = \beta$: symmetric around 0.5
- Large $\alpha, \beta$: concentrated (strong prior belief)

The animation below shows how the Beta PDF morphs as $\alpha$ and $\beta$ change — from uniform to concentrated, from skewed to symmetric.

In [ ]:
from manim import *

cfg.apply_manim_config()
math_text = cfg.math_text

from amstats.manim_utils import C

In [ ]:
%%manim -qm -v WARNING BetaPDFMorph


class BetaPDFMorph(Scene):
    """Animate the Beta PDF morphing through different (α, β) pairs."""

    def construct(self):
        from scipy.stats import beta as beta_dist

        title = Text("Beta(α, β) — PDF as parameters change", font_size=28).to_edge(UP)
        self.play(Write(title))

        axes = Axes(
            x_range=[0, 1, 0.2],
            y_range=[0, 4, 1],
            x_length=8,
            y_length=4.5,
            axis_config={"include_numbers": True, "font_size": 18},
        ).shift(DOWN * 0.3)
        x_lbl = axes.get_x_axis_label(
            Text("x", font_size=20), edge=DOWN, direction=DOWN
        )
        y_lbl = axes.get_y_axis_label(
            Text("f(x)", font_size=18), edge=LEFT, direction=LEFT
        )
        self.play(Create(axes), Write(x_lbl), Write(y_lbl), run_time=0.8)

        param_pairs = [(1, 1), (2, 5), (5, 2), (5, 5), (10, 10), (0.5, 0.5), (2, 2)]

        prev_curve = None
        prev_label = None

        for a, b in param_pairs:
            dist = beta_dist(a, b)
            curve = axes.plot(
                lambda x, d=dist: d.pdf(x),
                x_range=[0.01, 0.99],
                color=C.SALMON,
                stroke_width=3,
            )
            label = math_text(
                rf"\alpha = {a},\; \beta = {b}", font_size=24, color=C.GOLD
            ).to_corner(UR)

            if prev_curve is None:
                self.play(Create(curve), Write(label), run_time=0.8)
                prev_curve = curve
                prev_label = label
            else:
                self.play(
                    Transform(prev_curve, curve),
                    Transform(prev_label, label),
                    run_time=1,
                )
            self.wait(0.8)

        self.wait(1)

The Beta distribution is incredibly flexible: it can be uniform, U-shaped, skewed, or concentrated — all within $[0, 1]$. In Module 07 (Bayesian Inference) we'll use it as the **prior** for the probability parameter $p$ in a Binomial model: observing data will "update" the Beta shape from prior to posterior.

---

## 7. Summary: Distribution Gallery

| Distribution    | Support       | Parameters      | $E[X]$                        | $\text{Var}(X)$                                        | Use case                     |
|-----------------|---------------|-----------------|-------------------------------|--------------------------------------------------------|------------------------------|
| **Uniform**     | $[a, b]$      | $a, b$          | $\frac{a+b}{2}$               | $\frac{(b-a)^2}{12}$                                   | Equal likelihood on interval |
| **Normal**      | $\mathbb{R}$  | $\mu, \sigma^2$ | $\mu$                         | $\sigma^2$                                             | Measurement errors, CLT      |
| **Exponential** | $[0, \infty)$ | $\lambda$       | $1/\lambda$                   | $1/\lambda^2$                                          | Waiting times, memoryless    |
| **Gamma**       | $(0, \infty)$ | $\alpha, \beta$ | $\alpha/\beta$                | $\alpha/\beta^2$                                       | Generalises Exponential      |
| **Beta**        | $(0, 1)$      | $\alpha, \beta$ | $\frac{\alpha}{\alpha+\beta}$ | $\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ | Proportions, Bayesian priors |

### Relationships between distributions

- **Exponential** = Gamma with $\alpha = 1$
- **Chi-squared($n$)** = Gamma with $\alpha = n/2, \beta = 1/2$
- **Uniform(0,1)** = Beta with $\alpha = \beta = 1$
- Poisson events at rate $\lambda$ → inter-arrival times are Exponential($\lambda$)
- Binomial with large $n$ → approximated by Normal($np$, $np(1-p)$) (CLT — next notebook!)

---

## Exercises

**Exercise 2.1 (Uniform).** $X \sim \text{Uniform}(2, 8)$. Compute $P(3 \leq X \leq 5)$ analytically. Verify with `scipy.stats.uniform`.

**Exercise 2.2 (Normal).** Adult male heights are $\sim \mathcal{N}(175, 7^2)$ cm. What fraction are taller than 190 cm? Between 168 and 182 cm? Use both the 68-95-99.7 rule (approximately) and `scipy.stats.norm` (exactly).

**Exercise 2.3 (Exponential).** A server receives requests at rate $\lambda = 3$ per minute. What is $P(\text{wait} > 1 \text{ min})$? $E[\text{wait}]$? Verify with simulation.

**Exercise 2.4 (Gamma).** If individual service times are $\text{Exponential}(\lambda = 2)$, what is the distribution of the total time to serve 5 customers? Compute $E[X]$ and $\text{Var}(X)$.

**Exercise 2.5 (Beta).** Plot the Beta PDF for: (a) $\alpha = 1, \beta = 1$, (b) $\alpha = 0.5, \beta = 0.5$, (c) $\alpha = 10, \beta = 2$, (d) $\alpha = 100, \beta = 100$. Describe each shape in words.

**Exercise 2.6 (Challenge — Normal CDF).** There is no closed-form expression for $\Phi(z) = P(Z \leq z)$. Implement a from-scratch approximation using numerical integration (trapezoid rule) and compare with `scipy.stats.norm.cdf`.

---

## Key Takeaways

1. Continuous RVs use **PDFs** not PMFs. Probability = area under the curve, not the height.
2. The **Normal distribution** is the most important — it's characterised entirely by $\mu$ and $\sigma$, and the 68-95-99.7 rule gives quick probability estimates.
3. The **Exponential** distribution models memoryless waiting times and is the continuous analogue of the Geometric.
4. The **Gamma** generalises the Exponential; the **Beta** is defined on $[0, 1]$ and will be essential for Bayesian inference.
5. Distributions are connected in a web of relationships — understanding these connections is more valuable than memorising individual formulas.

**Next:** [03_central_limit_theorem.ipynb](03_central_limit_theorem.ipynb) — Why the Normal distribution appears everywhere: the Central Limit Theorem.

In [ ]:
cfg.save_gifs(clean=True)